In [43]:
from git import Repo 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import chroma
from langchain.chains import ConversationalRetrievalChain
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
import os


In [15]:
!mkdir -p test_repo

A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file test_repo already exists.
Error occurred while processing: test_repo.


In [16]:
repo_path= 'test_repo'                                                                         

In [17]:
from pathlib import Path

repo_path = Path("drinks-quality-prediction-system1").resolve()

if repo_path.exists():
    repo = Repo(str(repo_path))
    print("আগের repository ব্যবহার হচ্ছে:", repo_path)
else:
    repo = Repo.clone_from(
        "https://github.com/AKshihab/drinks-quality-prediction-system1",
        str(repo_path),
    )

আগের repository ব্যবহার হচ্ছে: C:\Users\shiha\Desktop\ML-Projects\source_code_analyzer\research\drinks-quality-prediction-system1


In [18]:
%pwd

'c:\\Users\\shiha\\Desktop\\ML-Projects\\source_code_analyzer\\research'

In [19]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*",
    parser=LanguageParser(language=Language.PYTHON,parser_threshold=500)
)

In [20]:
documnents = loader.load()

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf3 in position 13: invalid continuation byte

In [21]:
documnents

[]

In [27]:
from pathlib import Path
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser

repo_path = str(
    Path("drinks-quality-prediction-system1").resolve()
)

loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*.py",
    suffixes=[".py"],
    parser=LanguageParser(language="python"),
)

documents = loader.load()

print("Loaded documents:", len(documents))

if documents:
    print("First file:", documents[0].metadata)
    print(documents[0].page_content[:500])

Loaded documents: 65
First file: {'source': 'C:\\Users\\shiha\\Desktop\\ML-Projects\\source_code_analyzer\\research\\drinks-quality-prediction-system1\\app.py', 'content_type': 'functions_classes', 'language': 'python'}
def load_prediction_service(
    model_path: Path, metadata_path: Path
) -> PredictionService:
    return PredictionService(model_path, metadata_path)


In [28]:
documents

[Document(page_content='def load_prediction_service(\n    model_path: Path, metadata_path: Path\n) -> PredictionService:\n    return PredictionService(model_path, metadata_path)', metadata={'source': 'C:\\Users\\shiha\\Desktop\\ML-Projects\\source_code_analyzer\\research\\drinks-quality-prediction-system1\\app.py', 'content_type': 'functions_classes', 'language': 'python'}),
 Document(page_content='def load_reference_data(data_path: Path) -> pd.DataFrame:\n    return pd.read_csv(data_path)', metadata={'source': 'C:\\Users\\shiha\\Desktop\\ML-Projects\\source_code_analyzer\\research\\drinks-quality-prediction-system1\\app.py', 'content_type': 'functions_classes', 'language': 'python'}),
 Document(page_content='def main() -> None:\n    st.set_page_config(\n        page_title="Drinks Quality Prediction",\n        page_icon="🥤",\n        layout="wide",\n    )\n    st.title("Drinks Quality Prediction System")\n    st.write(\n        "Enter the measured properties of a drink and submit the f

In [29]:
documents[0]

Document(page_content='def load_prediction_service(\n    model_path: Path, metadata_path: Path\n) -> PredictionService:\n    return PredictionService(model_path, metadata_path)', metadata={'source': 'C:\\Users\\shiha\\Desktop\\ML-Projects\\source_code_analyzer\\research\\drinks-quality-prediction-system1\\app.py', 'content_type': 'functions_classes', 'language': 'python'})

In [30]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,chunk_size=500,chunk_overlap=20)

In [31]:
texts = documents_splitter.split_documents(documents)

In [32]:
len(texts)

162

In [36]:
from dotenv import load_dotenv,find_dotenv
load_dotenv(find_dotenv())
euri_api_key = os.environ.get("EURI_API_KEY")
print("Euri API configured:"),bool(euri_api_key)

Euri API configured:


(None, True)

In [40]:

embeddings = OpenAIEmbeddings(
    openai_api_key=euri_api_key,
    openai_api_base="https://api.euron.one/api/v1/euri",
    model="text-embedding-3-small",
    model_kwargs={"encoding_format": "float"},
)

# Test your key and model access.
vector = embeddings.embed_query("Hello, I am Shihab.")
print("Embedding dimensions:", len(vector))

Embedding dimensions: 1536


In [38]:
from langchain.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    openai_api_key=euri_api_key,
    openai_api_base="https://api.euron.one/api/v1/euri",
    model="text-embedding-3-small",
    model_kwargs={"encoding_format": "float"},
)

vector = embeddings.embed_query("Hello, I am Shihab.")
print("Embedding dimensions:", len(vector))

Embedding dimensions: 1536


In [50]:
from langchain.vectorstores import Chroma

vectordb = Chroma.from_documents(texts, embeddings, persist_directory="./data")

Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


In [ ]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=euri_api_key,
    openai_api_base="https://api.euron.one/api/v1/euri",
    model="gpt-4.1-mini",
    temperature=5,
)

In [54]:
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

In [55]:
qa=ConversationalRetrievalChain.from_llm(llm,retriever=vectordb.as_retriever(search_type="mmr",search_kwargs={"k": 5}),memory=memory)

In [60]:
question = "What is the purpose of class DataValidation in this project?"

In [61]:
result = qa(question)

In [62]:
result

{'question': 'What is the purpose of class DataValidation in this project?',
 'chat_history': [SystemMessage(content="The human asks about the purpose of the 'data_preprocessing.py' file in the project. The AI responds that there is no information or mention of that file in the provided context, so it does not know its specific purpose.", additional_kwargs={})],
 'answer': 'Based on the provided context, the class DataValidation in this project is responsible for validating data, likely as part of a machine learning pipeline. It is used to ensure that the input data meets certain criteria before further processing, such as training or prediction.\n\nSpecifically, the DataValidation class is part of the components of the project (as indicated by the import from mlproject.components.data_validation). It likely checks for the presence of required columns, validates schema conformity, handles missing values, and flags unexpected columns or invalid values. This is inferred from the presence

In [63]:
print(result['answer'])

Based on the provided context, the class DataValidation in this project is responsible for validating data, likely as part of a machine learning pipeline. It is used to ensure that the input data meets certain criteria before further processing, such as training or prediction.

Specifically, the DataValidation class is part of the components of the project (as indicated by the import from mlproject.components.data_validation). It likely checks for the presence of required columns, validates schema conformity, handles missing values, and flags unexpected columns or invalid values. This is inferred from the presence of configuration attributes like all_schema, target_column, allow_missing_values, and allow_unexpected_columns in the DataValidationConfig class, which probably configures how DataValidation operates.

Additionally, there is a snippet showing test cases where invalid values are tested to raise PredictionValidationError, suggesting DataValidation also plays a role in verifying